In [15]:
# https://pandas.pydata.org/
import pandas

# https://docs.python.org/3/library/math.html
import math

# https://numpy.org/
import numpy as np

# https://www.astropy.org/
from astropy import units as u
from astropy.time import Time
# https://docs.astropy.org/en/stable/coordinates/index.html
from astropy.coordinates import SkyCoord

# Selección

In [16]:
# https://vizier.cds.unistra.fr/viz-bin/VizieR-4
file = "data_clean/vizier_I_239_hip_main_LimpiezaNaNPlx.tsv"

# leer .csv
df = pandas.read_csv(
    file,              # nombre y ruta del archivo
    sep = '\t',        # símbolo de separación de columnas
    nrows = 118218,    # número de filas que se leerán
    index_col = 'HIP', # nombre de la columna índice
)

# dataframe
print( type(df) )

df.head()

<class 'pandas.core.frame.DataFrame'>


,recno,Proxy,RAhms,DEdms,Vmag,VarFlag,r_Vmag,RAICRS,DEICRS,AstroRef,...,BD,CoD,CPD,(V-I)red,SpType,r_SpType,HIPep,Erratum,_RA.icrs,_DE.icrs
HIP,,,,,,,,,,,,,,,,,,,,,
1,1,NaN,00 00 00.22,+01 05 20.4,9.10,NaN,H,0.000912,1.089013,NaN,...,B+00 5077,NaN,NaN,0.66,F5,S,HIPep,Erratum,0.000899,1.089009
2,2,NaN,00 00 00.91,-19 29 55.8,9.27,NaN,G,0.003797,-19.498837,+,...,B-20 6688,NaN,NaN,1.04,K3V,4,HIPep,Erratum,0.004265,-19.498840
3,3,NaN,00 00 01.20,+38 51 33.4,6.61,NaN,G,0.005008,38.859286,NaN,...,B+38 5108,NaN,NaN,0.00,B9,S,HIPep,Erratum,0.005024,38.859279
4,4,NaN,00 00 02.01,-51 53 36.8,8.06,NaN,H,0.008382,-51.893546,NaN,...,NaN,NaN,P-52 12237,0.43,F0V,2,HIPep,Erratum,0.008629,-51.893546
5,5,NaN,00 00 02.39,-40 35 28.4,8.55,NaN,H,0.009965,-40.591224,NaN,...,NaN,C-41 15372,P-41 9991,0.95,G8III,2,HIPep,Erratum,0.009973,-40.591202


## Métodos

In [17]:
#
def calculate_abs_mag(apparent_magnitude, milliarcsecond):
    """ 
        Calcula la magnitud absoluta de la forma: 
        
        M = m + 5 + 5 log π"

        M = (log π" * 5) + 5 + m

        El promedio de logaritmo 10 de pi doble prima (paralaje) y 5, más 
        la suma de 5 y la magnitud aparente es igual a la magnitud absoluta.
        
        @params {Series} apparent_magnitude: el brillo a simple vista
        @params {Series} milliarcsecond: la distancia en mas
        @returns {Series} absolute_magnitude
    """
    arcsecond = milliarcsecond / 1000
    absolute_magnitude = ( np.log10( arcsecond ) * 5 ) + 5 + apparent_magnitude
    
    return absolute_magnitude

def calculate_distance(apparent_magnitude, absolute_magnitude):
    """ Calcula  distancia de la forma: 
        
        d = 10**((m - M + 5) / 5)
        
        La diferencia de las magnitudes más cinco sobre cinco, 
        por logaritmo de 10 es igual a la distancia en pársecs.
        
        @param {Series} apparent_magnitude: el brillo a simple vista
        @param {Series} absolute_magnitude: el brillo real
        @return {Series} parsecs
    """
    logarithm = ( apparent_magnitude - absolute_magnitude  + 5) / 5
    parsecs = 10**logarithm
    
    return parsecs

""" De la forma: m - M = 5 log d - 5

    La diferencia de las magnitudes es igual a la diferencia 
    de cinco menos cinco por el logaritmo 10 de la distancia.
"""


' De la forma: m - M = 5 log d - 5\n\n    La diferencia de las magnitudes es igual a la diferencia \n    de cinco menos cinco por el logaritmo 10 de la distancia.\n'

In [18]:
# copia del dataframe con la selección de columnas
df_selection = df[['RAhms','DEdms','Vmag', 'RAICRS', 'DEICRS', 'Plx']].copy()

## Calcular Magnitud Absoluta

In [19]:
absolute_magnitude = calculate_abs_mag(df['Vmag'], df['Plx'])
absolute_magnitude

HIP
1         1.845016
2         5.972221
3        -1.146468
4         2.506509
5         0.839409
            ...   
118318   -1.593494
118319    3.362666
118320    1.084850
118321    5.618767
118322   -0.809909
Length: 113710, dtype: float64

In [20]:
# Add a new column name ABSmag with the absolute magnitude calculated
df_selection['ABSmag'] = calculate_abs_mag(df_selection['Vmag'], df_selection['Plx'])
df_selection['ABSmag'].describe()

count    113710.000000
mean          1.691757
std           2.412337
min         -13.310000
25%           0.287908
50%           1.559351
75%           3.143711
max          15.449015
Name: ABSmag, dtype: float64

## Calcular distancia

Aquí se puede calcular la distancia y transformarla a unidades pársec de distintas maneras:

- 1000.0 / dataframe['Plx']
- Utilizar el módulo distancia de la forma: d = 10**((m - M + 5) / 5)
- Utilizar el módulo distancia de la forma: m - M = 5log * d - 5

### Calcular Distancia en pársecs

In [21]:
# Add a new column name ABSmag with the absolute magnitude calculated
parsecs = calculate_distance( df_selection['Vmag'], df_selection['ABSmag'] )
parsecs

HIP
1         282.485876
2          45.662100
3         355.871886
4         129.032258
5         348.432056
             ...    
118318    520.833333
118319     94.073377
118320    200.000000
118321     52.029136
118322    114.810563
Length: 113710, dtype: float64

In [22]:
# Add a new column name Pc with the distance in pársec calculated
df_selection['Pc'] = calculate_distance( df_selection['Vmag'], df_selection['ABSmag'] )
df_selection['Pc'].describe()

count    113710.000000
mean        453.601423
std        2316.921437
min           1.294783
25%         115.874855
50%         208.768267
75%         367.647059
max      100000.000000
Name: Pc, dtype: float64

### Transformar la distancia de pársecs a años luz

In [23]:
lightyears = df_selection["Pc"] * 3.261598
lightyears

HIP
1          921.355367
2          148.931416
3         1160.711032
4          420.851355
5         1136.445296
             ...     
118318    1698.748958
118319     306.829539
118320     652.319600
118321     169.698127
118322     374.465901
Name: Pc, Length: 113710, dtype: float64

In [24]:
# Add a new column name Ly with the distance transformed from pársecs to lightyear
df_selection["Ly"] = df_selection["Pc"] * 3.261598
df_selection["Ly"].describe()

count    113710.000000
mean       1479.465494
std        7556.866324
min           4.223063
25%         377.937196
50%         680.918163
75%        1199.116912
max      326159.800000
Name: Ly, dtype: float64

In [25]:
df_selection.head(2)

,RAhms,DEdms,Vmag,RAICRS,DEICRS,Plx,ABSmag,Pc,Ly
HIP,,,,,,,,,
1,00 00 00.22,+01 05 20.4,9.10,0.000912,1.089013,3.54,1.845016,282.485876,921.355367
2,00 00 00.91,-19 29 55.8,9.27,0.003797,-19.498837,21.90,5.972221,45.662100,148.931416


## Convertir las coordenas ecuatoriales a coordenadas cartesianas

In [26]:

ra_series = df_selection["RAICRS"]
dec_series = df_selection["DEICRS"]
pc_series = df_selection["Pc"]

# Convertir a array antes de pasarlo a SkyCoord
ra = np.asarray(ra_series) * u.deg
dec = np.asarray(dec_series) * u.deg
pc = np.asarray(pc_series) * u.pc

coordinates = SkyCoord(ra=ra, dec=dec, distance=pc, frame="icrs").cartesian
print(len(coordinates))
print(coordinates)

113710
[(282.43485164,  0.00449489,    5.36884849),
 ( 43.04329963,  0.00285276,  -15.24144898),
 (277.11358524,  0.02422117,  223.27753944), ...,
 (198.92012642, -0.08312395,   20.75515344),
 ( 22.50350573, -0.00854689,  -46.91080028),
 ( 47.47057459, -0.01757125, -104.53712029)] pc


### X

In [27]:
print(type(coordinates.x))
coordinates.x

<class 'astropy.units.quantity.Quantity'>


<Quantity [282.43485164,  43.04329963, 277.11358524, ..., 198.92012642,
            22.50350573,  47.47057459] pc>

In [28]:
pandas.Series(coordinates.x.value)

0         282.434852
1          43.043300
2         277.113585
3          79.628969
4         264.589185
             ...    
113705    510.060190
113706     86.957517
113707    198.920126
113708     22.503506
113709     47.470575
Length: 113710, dtype: float64

In [29]:
# transformando las unidades Quantity de astropy a serie de pandas con el index del df
df_selection['X'] = pandas.Series(coordinates.x.value, index=df_selection.index)
df_selection.head(2)

,RAhms,DEdms,Vmag,RAICRS,DEICRS,Plx,ABSmag,Pc,Ly,X
HIP,,,,,,,,,,
1,00 00 00.22,+01 05 20.4,9.10,0.000912,1.089013,3.54,1.845016,282.485876,921.355367,282.434852
2,00 00 00.91,-19 29 55.8,9.27,0.003797,-19.498837,21.90,5.972221,45.662100,148.931416,43.043300


### Y

In [30]:
print(type(coordinates.y))
print(coordinates.y)
pandas.Series(coordinates.y.value)

<class 'astropy.units.quantity.Quantity'>
[ 0.00449489  0.00285276  0.02422117 ... -0.08312395 -0.00854689
 -0.01757125] pc


0         0.004495
1         0.002853
2         0.024221
3         0.011649
4         0.046019
            ...   
113705   -0.322761
113706   -0.039593
113707   -0.083124
113708   -0.008547
113709   -0.017571
Length: 113710, dtype: float64

In [31]:
df_selection['Y'] = pandas.Series(coordinates.y.value, index=df_selection.index)
df_selection.head(2)

,RAhms,DEdms,Vmag,RAICRS,DEICRS,Plx,ABSmag,Pc,Ly,X,Y
HIP,,,,,,,,,,,
1,00 00 00.22,+01 05 20.4,9.10,0.000912,1.089013,3.54,1.845016,282.485876,921.355367,282.434852,0.004495
2,00 00 00.91,-19 29 55.8,9.27,0.003797,-19.498837,21.90,5.972221,45.662100,148.931416,43.043300,0.002853


### Z

In [32]:
print(type(coordinates.z))
print(coordinates.z)
pandas.Series(coordinates.z.value)

<class 'astropy.units.quantity.Quantity'>
[   5.36884849  -15.24144898  223.27753944 ...   20.75515344  -46.91080028
 -104.53712029] pc


0           5.368848
1         -15.241449
2         223.277539
3        -101.531034
4        -226.710076
             ...    
113705    105.384343
113706    -35.891351
113707     20.755153
113708    -46.910800
113709   -104.537120
Length: 113710, dtype: float64

In [33]:
df_selection['Z'] = pandas.Series(coordinates.z.value, index=df_selection.index)
df_selection.head(2)

,RAhms,DEdms,Vmag,RAICRS,DEICRS,Plx,ABSmag,Pc,Ly,X,Y,Z
HIP,,,,,,,,,,,,
1,00 00 00.22,+01 05 20.4,9.10,0.000912,1.089013,3.54,1.845016,282.485876,921.355367,282.434852,0.004495,5.368848
2,00 00 00.91,-19 29 55.8,9.27,0.003797,-19.498837,21.90,5.972221,45.662100,148.931416,43.043300,0.002853,-15.241449


In [34]:
#df_selection[['RAhms','DEdms','Vmag', 'Plx', 'ABSmag', 'Pc', 'Ly', 'X', 'Y', 'Z']]
df_selection.drop(columns=['RAICRS', 'DEICRS'])

,RAhms,DEdms,Vmag,Plx,ABSmag,Pc,Ly,X,Y,Z
HIP,,,,,,,,,,
1,00 00 00.22,+01 05 20.4,9.10,3.54,1.845016,282.485876,921.355367,282.434852,0.004495,5.368848
2,00 00 00.91,-19 29 55.8,9.27,21.90,5.972221,45.662100,148.931416,43.043300,0.002853,-15.241449
3,00 00 01.20,+38 51 33.4,6.61,2.81,-1.146468,355.871886,1160.711032,277.113585,0.024221,223.277539
4,00 00 02.01,-51 53 36.8,8.06,7.75,2.506509,129.032258,420.851355,79.628969,0.011649,-101.531034
5,00 00 02.39,-40 35 28.4,8.55,2.87,0.839409,348.432056,1136.445296,264.589185,0.046019,-226.710076
...,...,...,...,...,...,...,...,...,...,...
118318,23 59 51.30,+11 40 25.4,6.99,1.92,-1.593494,520.833333,1698.748958,510.060190,-0.322761,105.384343
118319,23 59 53.74,-22 25 41.4,8.23,10.63,3.362666,94.073377,306.829539,86.957517,-0.039593,-35.891351
118320,23 59 54.25,+05 57 23.9,7.59,5.00,1.084850,200.000000,652.319600,198.920126,-0.083124,20.755153


## Filtrar registros por distancia

In [35]:
#df_filter = df_selection[['RAhms','DEdms','Vmag', 'Plx', 'ABSmag', 'Pc', 'Ly', 'X', 'Y', 'Z']].copy()
df_filter = df_selection.drop(columns=['RAICRS', 'DEICRS']).copy()

### Pc < 1000

In [36]:
# dame las estrellas menores a 1000 pársecs de distancia de menor a mayor
df_filter[df_filter['Pc'] < 1000].sort_values(['Pc'], ascending = True)

,RAhms,DEdms,Vmag,Plx,ABSmag,Pc,Ly,X,Y,Z
HIP,,,,,,,,,,
70890,14 29 47.75,-62 40 52.9,11.01,772.33,15.449015,1.294783,4.223063,-0.471754,-0.361322,-1.150373
71681,14 39 39.39,-60 50 22.1,1.35,742.12,5.702371,1.347491,4.394974,-0.503598,-0.421285,-1.176707
71683,14 39 40.90,-60 50 06.5,-0.01,742.12,4.342371,1.347491,4.394974,-0.503620,-0.421397,-1.176658
87937,17 57 48.97,+04 40 05.8,9.54,549.01,13.237901,1.821460,5.940872,-0.017299,-1.815335,0.148243
54035,11 03 20.61,+35 58 53.3,7.49,392.40,10.458645,2.548420,8.311922,-1.999506,0.504621,1.497257
...,...,...,...,...,...,...,...,...,...,...
117822,23 53 44.17,-29 23 47.7,7.42,1.01,-2.558393,990.099010,3229.304950,862.294723,-23.573081,-485.991945
91066,18 34 32.76,-24 13 20.6,6.53,1.01,-3.448393,990.099010,3229.304950,135.588819,-892.692143,-406.217256
94463,19 13 35.48,-11 13 44.9,9.44,1.01,-0.538393,990.099010,3229.304950,306.506171,-921.507548,-192.805224


### Polaris, Sirius, Proxima Centauri

In [37]:
hip_polaris = 11767
hip_sirius = 32349
hip_proxima = 70890

# Polaris se localiza a 132 pársecs con una magnitud absoluta de -3.6
df_filter[df_filter['Pc'] < 1000].sort_values(['Pc'], ascending = True).loc[hip_polaris]

RAhms     02 31 47.08
DEdms     +89 15 50.9
Vmag             1.97
Plx              7.56
ABSmag      -3.637391
Pc         132.275132
Ly         431.428307
X            1.339648
Y            1.044622
Z          132.264223
Name: 11767, dtype: object

In [38]:
# Sirius y Proxima Centauri se localizan a 2.6 y 1.29 pársecs
df_filter[df_filter['Pc'] < 1000].sort_values(['Pc'], ascending = True).loc[[hip_polaris, hip_sirius, hip_proxima]]

,RAhms,DEdms,Vmag,Plx,ABSmag,Pc,Ly,X,Y,Z
HIP,,,,,,,,,,
11767,02 31 47.08,+89 15 50.9,1.97,7.56,-3.637391,132.275132,431.428307,1.339648,1.044622,132.264223
32349,06 45 09.25,-16 42 47.3,-1.44,379.21,1.454399,2.637061,8.601034,-0.494399,2.476801,-0.758367
70890,14 29 47.75,-62 40 52.9,11.01,772.33,15.449015,1.294783,4.223063,-0.471754,-0.361322,-1.150373


### Pc < 100

In [39]:
# dame las estrellas menores a 100 pársecs de distancia de menor a mayor
df_filter[df_filter['Pc'] < 100].sort_values(['Pc'], ascending = True).shape

(22951, 10)

### Pc < 10

In [40]:
# dame las estrellas menores a 10 pársecs de distancia de menor a mayor
df_filter[df_filter['Pc'] < 10].sort_values(['Pc'], ascending = True)

,RAhms,DEdms,Vmag,Plx,ABSmag,Pc,Ly,X,Y,Z
HIP,,,,,,,,,,
70890,14 29 47.75,-62 40 52.9,11.01,772.33,15.449015,1.294783,4.223063,-0.471754,-0.361322,-1.150373
71683,14 39 40.90,-60 50 06.5,-0.01,742.12,4.342371,1.347491,4.394974,-0.503620,-0.421397,-1.176658
71681,14 39 39.39,-60 50 22.1,1.35,742.12,5.702371,1.347491,4.394974,-0.503598,-0.421285,-1.176707
87937,17 57 48.97,+04 40 05.8,9.54,549.01,13.237901,1.821460,5.940872,-0.017299,-1.815335,0.148243
54035,11 03 20.61,+35 58 53.3,7.49,392.40,10.458645,2.548420,8.311922,-1.999506,0.504621,1.497257
...,...,...,...,...,...,...,...,...,...,...
71898,14 42 22.01,+66 03 21.2,10.88,101.34,10.908904,9.867772,32.184705,-3.041098,-2.605775,9.018569
84581,17 17 23.46,-07 52 36.6,11.09,100.89,11.109241,9.911785,32.328258,-1.814887,-9.649070,-1.358352
102409,20 45 09.34,-31 20 24.1,8.81,100.59,8.822774,9.941346,32.424674,5.602749,-6.379963,-5.170652


In [41]:
df_10pc = df_filter[df_filter['Pc'] < 10]
df_10pc.shape

(182, 10)

## exportar

In [42]:
# exporta el dataframe a .csv, .tsv , .json
out_file = "data_clean/vizier_I_239_hip_main_10pc"

#df_10pc.to_csv(out_file)
#df_10pc.to_json(out_file + ".tsv", sep='\t')
#df_10pc.to_json(out_file + ".json", orient="table")